# Customer revenue pipeline — postprocessing

Compare model metrics, choose the winner, and produce the analyst-facing report.

In [ ]:
# @node: Compare models  [transform]  in=baseline_scores<-model_baseline:Train baseline.baseline_scores,segmented_scores<-model_advanced:Train segmented model.segmented_scores  out=comparison
import pandas as pd

comparison = (
    pd.concat([baseline_scores, segmented_scores], ignore_index=True)
    .sort_values("rmse")
    .reset_index(drop=True)
)
display(comparison.round(2))

In [ ]:
# @node: Build forecast package  [transform]  in=comparison<-Compare models.comparison,baseline_predictions<-model_baseline:Train baseline.baseline_predictions,segmented_predictions<-model_advanced:Train segmented model.segmented_predictions  out=winning_predictions
winner = comparison.iloc[0]["model"]
winning_predictions = (
    segmented_predictions.copy()
    if winner == "segmented by channel"
    else baseline_predictions.copy()
)
winning_predictions["absolute_error"] = (winning_predictions["prediction"] - winning_predictions["revenue"]).abs()
display(winning_predictions.head(10))

In [ ]:
# @node: Analyst report  [output]  in=comparison<-Compare models.comparison,winning_predictions<-Build forecast package.winning_predictions
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

best = comparison.iloc[0]
print(f"Recommended model: {best['model']} | RMSE: {best['rmse']:.2f} | MAE: {best['mae']:.2f}")
by_channel = (
    winning_predictions.groupby("channel", as_index=False)
    .agg(actual=("revenue", "sum"), predicted=("prediction", "sum"), mae=("absolute_error", "mean"))
    .sort_values("actual", ascending=False)
)
display(by_channel.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(winning_predictions["revenue"], winning_predictions["prediction"], alpha=0.65, color="#2563eb")
limit = max(winning_predictions["revenue"].max(), winning_predictions["prediction"].max())
ax.plot([0, limit], [0, limit], color="#111827", linestyle="--", linewidth=1)
ax.set_title("Actual vs predicted revenue")
ax.set_xlabel("Actual revenue")
ax.set_ylabel("Predicted revenue")
fig.tight_layout()